In [ ]:
import math
from typing import List, Optional, Tuple

import torch
import torch.nn as nn
import torch.nn.functional as F


# ---------------------------------------------------------------------------
# Building blocks
# ---------------------------------------------------------------------------

class ResidualBlock(nn.Module):
    """
    Fully-connected residual block (Eq. 13).

      x_{l+1} = F(x_l) + x_l
      F(x_l)  = BN(Linear(ReLU(BN(Linear(x_l)))))

    A projection shortcut is added when in_dim ≠ out_dim so that the
    skip connection always has compatible dimensions.
    """

    def __init__(
        self,
        in_dim: int,
        out_dim: int,
        dropout: float = 0.0,
        use_bn: bool = True,
    ):
        super().__init__()
        self.use_bn = use_bn

        self.linear1 = nn.Linear(in_dim, out_dim)
        self.bn1 = nn.BatchNorm1d(out_dim) if use_bn else nn.Identity()
        self.act = nn.ReLU(inplace=True)
        self.dropout = nn.Dropout(p=dropout)

        self.linear2 = nn.Linear(out_dim, out_dim)
        self.bn2 = nn.BatchNorm1d(out_dim) if use_bn else nn.Identity()

        # Projection shortcut when dimensions differ
        if in_dim != out_dim:
            self.shortcut = nn.Sequential(
                nn.Linear(in_dim, out_dim, bias=False),
                nn.BatchNorm1d(out_dim) if use_bn else nn.Identity(),
            )
        else:
            self.shortcut = nn.Identity()

        self._init_weights()

    def _init_weights(self) -> None:
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, nonlinearity="relu")
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        identity = self.shortcut(x)

        out = self.linear1(x)
        out = self.bn1(out)
        out = self.act(out)
        out = self.dropout(out)

        out = self.linear2(out)
        out = self.bn2(out)

        out = out + identity
        out = self.act(out)
        return out


class SelfAttentionLayer(nn.Module):
    """
    Feature-level self-attention (Eq. 14–15).

      A   = softmax(X · X^T)          shape: (N, N)  — patient-wise
      X_O = A · X                     shape: (N, d)

    Note: the paper applies attention over the *feature dimension* of the
    batch.  We implement it as described (X_I @ X_I^T ∈ R^{N×N}) so that
    each sample's output is a weighted mixture of all samples' features.
    During inference with a single sample the attention degenerates to the
    identity; for batch inference it reweights samples by mutual similarity.
    """

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (N, d)
        scores = x @ x.T                         # (N, N)
        # Scale by sqrt(d) for numerical stability (à la scaled dot-product)
        d = x.shape[-1]
        scores = scores / math.sqrt(d)
        A = F.softmax(scores, dim=-1)            # (N, N)
        out = A @ x                              # (N, d)
        return out


# ---------------------------------------------------------------------------
# Full ResDeepSurv model
# ---------------------------------------------------------------------------

class ResDeepSurv(nn.Module):
    """
    ResDeepSurv: residual + self-attention survival network.

    Parameters
    ----------
    input_dim       : number of input features (after radiomics extraction).
    hidden_dims     : list of hidden dimensions for each residual block.
    num_res_blocks  : number of residual blocks (overrides len(hidden_dims)
                      if both are given; hidden_dims is cycled if shorter).
    dropout         : dropout probability inside residual blocks.
    use_bn          : use BatchNorm in residual blocks.

    Forward pass
    ------------
    Returns a (N,) risk score tensor θ^T x (i.e. the log-hazard ratio)
    that feeds directly into the Cox partial likelihood.
    """

    def __init__(
        self,
        input_dim: int,
        hidden_dims: List[int] = (256, 256, 128),
        num_res_blocks: int = 3,
        dropout: float = 0.3,
        use_bn: bool = True,
    ):
        super().__init__()
        self.input_dim = input_dim
        self.hidden_dims = hidden_dims
        self.num_res_blocks = num_res_blocks

        # ── Input projection layer ──────────────────────────────────────
        first_hidden = hidden_dims[0]
        self.input_layer = nn.Sequential(
            nn.Linear(input_dim, first_hidden),
            nn.BatchNorm1d(first_hidden) if use_bn else nn.Identity(),
            nn.ReLU(inplace=True),
        )

        # ── Residual blocks ─────────────────────────────────────────────
        blocks: List[nn.Module] = []
        dims = list(hidden_dims)
        # Extend dims list to cover num_res_blocks
        while len(dims) < num_res_blocks + 1:
            dims.append(dims[-1])

        for i in range(num_res_blocks):
            blocks.append(
                ResidualBlock(
                    in_dim=dims[i],
                    out_dim=dims[i + 1],
                    dropout=dropout,
                    use_bn=use_bn,
                )
            )
        self.res_blocks = nn.ModuleList(blocks)

        # ── Self-attention ───────────────────────────────────────────────
        self.attention = SelfAttentionLayer()

        # ── Output head: scalar risk score ──────────────────────────────
        self.output_layer = nn.Linear(dims[num_res_blocks], 1)
        nn.init.xavier_uniform_(self.output_layer.weight)
        nn.init.zeros_(self.output_layer.bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Parameters
        ----------
        x : (N, input_dim) feature tensor.

        Returns
        -------
        risk : (N,) log-hazard ratio θ^T x.
        """
        out = self.input_layer(x)

        for block in self.res_blocks:
            out = block(out)

        out = self.attention(out)                    # (N, d_last)
        risk = self.output_layer(out).squeeze(-1)    # (N,)
        return risk

    # ------------------------------------------------------------------
    # Convenience
    # ------------------------------------------------------------------

    def predict_risk(
        self,
        x: torch.Tensor,
        device: Optional[torch.device] = None,
    ) -> torch.Tensor:
        """Inference-mode forward (no grad, moves to device if given)."""
        if device is not None:
            x = x.to(device)
            self.to(device)
        self.eval()
        with torch.no_grad():
            return self.forward(x)

    def count_parameters(self) -> int:
        return sum(p.numel() for p in self.parameters() if p.requires_grad)


# ---------------------------------------------------------------------------
# Cox partial likelihood loss (Eq. 18)
# ---------------------------------------------------------------------------

class CoxPHLoss(nn.Module):
    """
    Negative Cox partial log-likelihood (Eq. 17–18 in the paper).

      l = -Σ_{i: E_i=1} [ risk_i - log Σ_{j ∈ R(y_i)} exp(risk_j) ]

    where R(y_i) = { j : y_j ≥ y_i } is the risk set at time y_i.

    Implementation note
    -------------------
    We use the Breslow tie-handling approximation (standard for DL):
      log Σ_{j: y_j ≥ y_i} exp(risk_j)

    The batch is assumed to be *sorted in descending order of observed time*
    before calling forward(); the SurvivalDataset / collate_fn handles this.
    """

    def forward(
        self,
        risk: torch.Tensor,           # (N,) predicted log-hazard
        times: torch.Tensor,          # (N,) observed times
        events: torch.Tensor,         # (N,) 1=event, 0=censored
    ) -> torch.Tensor:
        """
        Parameters
        ----------
        risk   : (N,) float — output of ResDeepSurv.forward().
        times  : (N,) float — survival / censoring times.
        events : (N,) float/int — event indicator (1 = event).

        Returns
        -------
        Scalar loss tensor.
        """
        # Sort descending by time (needed for cumulative risk-set log-sum-exp)
        sort_idx = torch.argsort(times, descending=True)
        risk   = risk[sort_idx]
        events = events[sort_idx].bool()

        if events.sum() == 0:
            # No observed events in this batch — return zero loss
            return torch.tensor(0.0, requires_grad=True, device=risk.device)

        # Log-sum-exp of cumulative risk set (numerically stable)
        # cumulative_logsumexp[i] = log Σ_{j ≤ i} exp(risk_j)  (descending sort)
        log_cumsum = _log_cumsum_exp(risk)   # (N,)

        # Partial log-likelihood for event patients only
        uncensored_loss = risk[events] - log_cumsum[events]
        loss = -uncensored_loss.mean()
        return loss


def _log_cumsum_exp(x: torch.Tensor) -> torch.Tensor:
    """
    Numerically stable log of the cumulative sum of exp(x).

      out[i] = log( Σ_{j=0}^{i} exp(x[j]) )

    Uses the standard log-sum-exp trick:
      max_val + log( Σ exp(x - max_val) )
    applied cumulatively.
    """
    # Running max for numerical stability
    max_val, _ = torch.cummax(x, dim=0)
    exp_shifted = torch.exp(x - max_val)
    log_cumsum = torch.log(torch.cumsum(exp_shifted, dim=0)) + max_val
    return log_cumsum

In [ ]:
import pandas as pd

train_df = pd.read_csv("train_engineered_features.csv")
test_df = pd.read_csv("test_engineered_features.csv")
len(train_df), len(test_df)

(187, 47)

In [ ]:
len(train_df.columns)

56

In [ ]:
train_df.columns

Index(['BraTS20ID', 'Survival_days',
       'FLAIR_WT_original_shape_Maximum2DDiameterColumn',
       'FLAIR_WT_original_shape_Maximum2DDiameterSlice',
       'FLAIR_WT_original_shape_Sphericity',
       'FLAIR_WT_original_glrlm_RunLengthNonUniformity',
       'FLAIR_WT_original_glszm_LargeAreaEmphasis',
       'FLAIR_WT_original_glszm_LargeAreaHighGrayLevelEmphasis',
       'T1ce_WT_original_firstorder_Kurtosis',
       'T1ce_WT_original_firstorder_Mean',
       'T1ce_WT_original_glrlm_LongRunLowGrayLevelEmphasis',
       'T1ce_WT_original_glrlm_RunLengthNonUniformity',
       'T1ce_WT_original_glszm_SmallAreaLowGrayLevelEmphasis',
       'T1ce_WT_log-sigma-2-0-mm-3D_glszm_GrayLevelNonUniformity',
       'T1ce_WT_log-sigma-2-0-mm-3D_glszm_ZoneEntropy', 'volume_TC',
       'FLAIR_TC_original_shape_Flatness',
       'FLAIR_TC_original_shape_Maximum2DDiameterSlice',
       'FLAIR_TC_original_glszm_HighGrayLevelZoneEmphasis',
       'FLAIR_TC_log-sigma-2-0-mm-3D_glcm_ClusterProminence',
 

In [ ]:
import wandb
from google.colab import userdata

try:
    wandb_key = userdata.get('WANDB')
    wandb.login(key=wandb_key)
    print("Successfully logged into Weights & Biases!")
except userdata.SecretNotFoundError:
    print("WANDB_API_KEY not found in Colab secrets. Please add it to use wandb.")

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: WARNING [wandb.login()] Changing session credentials to explicit value for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc


Successfully logged into Weights & Biases!


In [ ]:
!pip install -q scikit-survival

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
import numpy as np

class SurvivalDataset(Dataset):
    """
    PyTorch Dataset for survival analysis.

    Stores feature matrix X, observed times T, and event indicators E as
    float32 tensors so they can be consumed directly by a DataLoader.

    Parameters
    ----------
    features : np.ndarray, shape (N, d)
        Patient feature matrix (e.g. radiomics + optional clinical vars).
    times : np.ndarray, shape (N,)
        Observed survival or censoring times (positive floats).
    events : np.ndarray, shape (N,)
        Event indicator: 1 = event occurred, 0 = right-censored.
    """

    def __init__(
        self,
        features: np.ndarray,
        times: np.ndarray,
        events: np.ndarray,
    ) -> None:
        if features.shape[0] != len(times) or features.shape[0] != len(events):
            raise ValueError(
                f"features ({features.shape[0]}), times ({len(times)}), and "
                f"events ({len(events)}) must all have the same length."
            )
        self.X = torch.tensor(features, dtype=torch.float32)
        self.T = torch.tensor(times,    dtype=torch.float32)
        self.E = torch.tensor(events,   dtype=torch.float32)

    def __len__(self) -> int:
        return len(self.T)

    def __getitem__(
        self, idx: int
    ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        """Return (features, time, event) for patient at index idx."""
        return self.X[idx], self.T[idx], self.E[idx]


def survival_collate(
    batch: List[Tuple[torch.Tensor, torch.Tensor, torch.Tensor]],
) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """
    Custom collate function for survival DataLoaders.

    The Cox partial likelihood loss (Eq. 18 in the paper) requires the
    batch to be **sorted in descending order of observed time** so the
    cumulative risk-set log-sum-exp can be computed with a single cumsum
    pass.  This collate function re-sorts every mini-batch after stacking,
    regardless of the DataLoader's shuffle setting.

    Parameters
    ----------
    batch : list of (X_i, T_i, E_i) tuples
        Raw samples returned by SurvivalDataset.__getitem__.

    Returns
    -------
    X : torch.Tensor, shape (B, d)  — features, sorted desc by time.
    T : torch.Tensor, shape (B,)    — times,    sorted desc.
    E : torch.Tensor, shape (B,)    — events,   sorted desc.
    """
    X = torch.stack([item[0] for item in batch])   # (B, d)
    T = torch.stack([item[1] for item in batch])   # (B,)
    E = torch.stack([item[2] for item in batch])   # (B,)

    # Descending sort by observed time — required by CoxPHLoss
    sort_idx = torch.argsort(T, descending=True)
    return X[sort_idx], T[sort_idx], E[sort_idx]

In [ ]:
import copy
import numpy as np
import torch
from torch.utils.data import DataLoader
from sklearn.model_selection import KFold
import torch.optim as optim
from sksurv.metrics import concordance_index_censored, integrated_brier_score
import wandb
import time
# ---------------------------------------------------------------------------
# Metrics Helpers
# ---------------------------------------------------------------------------

def compute_breslow_baseline_hazard(train_risks, train_times, train_events):
    """Estimate baseline cumulative hazard using Breslow's method."""
    sort_idx = np.argsort(train_times)
    t = train_times[sort_idx]
    e = train_events[sort_idx]
    r = train_risks[sort_idx]

    # Clip risks to avoid exp overflow
    clipped_r = np.clip(r, -20.0, 20.0)
    exp_r = np.exp(clipped_r)

    unique_times = np.unique(t[e == 1])
    if len(unique_times) == 0:
        return np.array([t.min()]), np.array([0.0])

    h0 = np.zeros_like(unique_times, dtype=float)
    for i, time_val in enumerate(unique_times):
        risk_set = t >= time_val
        events_at_t = np.sum(e[t == time_val])
        denom = np.sum(exp_r[risk_set])
        h0[i] = events_at_t / denom if denom > 0 else 0.0

    return unique_times, np.cumsum(h0)


def predict_survival_functions(test_risks, breslow_times, breslow_H0, eval_times):
    """Convert risk scores to survival probabilities at eval_times."""
    idx = np.searchsorted(breslow_times, eval_times, side="right") - 1
    H0_eval = np.where(idx >= 0, breslow_H0[idx], 0)
    # Clip risk to avoid overflow in S(t|x) = exp(-H0(t) * exp(risk))
    clipped_risks = np.clip(test_risks, -20.0, 20.0)
    return np.exp(-H0_eval.reshape(1, -1) * np.exp(clipped_risks).reshape(-1, 1))


def train_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    total_loss = 0.0
    for X_batch, times_batch, events_batch in dataloader:
        X_batch = X_batch.to(device)
        times_batch = times_batch.to(device)
        events_batch = events_batch.to(device)

        optimizer.zero_grad()
        risk_scores = model(X_batch)
        loss = criterion(risk_scores, times_batch, events_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    return total_loss / len(dataloader)


def evaluate(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0.0
    all_risks, all_times, all_events = [], [], []

    with torch.no_grad():
        for X_batch, times_batch, events_batch in dataloader:
            X_batch = X_batch.to(device)
            times_batch = times_batch.to(device)
            events_batch = events_batch.to(device)

            risk_scores = model(X_batch)
            loss = criterion(risk_scores, times_batch, events_batch)
            total_loss += loss.item()

            all_risks.append(risk_scores.cpu().numpy())
            all_times.append(times_batch.cpu().numpy())
            all_events.append(events_batch.cpu().numpy())

    return (
        total_loss / len(dataloader),
        np.concatenate(all_risks),
        np.concatenate(all_times),
        np.concatenate(all_events),
    )


def run_cross_validation(
    X_train_cv, times_train_cv, events_train_cv,
    X_test, times_test, events_test, input_dim,
    num_folds=5, epochs=20, batch_size=32, lr=1e-4,
):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    criterion = CoxPHLoss()
    events_train_cv_bool = events_train_cv.astype(bool)

    test_dataset = SurvivalDataset(X_test, times_test, events_test.astype(np.float32))
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    y_test_sksurv = np.array(
        list(zip(events_test, times_test)),
        dtype=[("Status", "?"), ("Survival_in_days", "<f8")],
    )

    cv_group = f"5fold_cv_{int(time.time())}"
    kf = KFold(n_splits=num_folds, shuffle=True, random_state=42)

    # ── Store all fold models and their CV risks for ensemble ────────────
    fold_model_states = []   # best state_dict from each fold
    fold_cv_risks     = []   # risks on the full CV set from each fold model
    fold_cv_times     = None
    fold_cv_events    = None

    for fold, (train_idx, val_idx) in enumerate(kf.split(X_train_cv)):
        print(f"\n--- Starting Fold {fold + 1}/{num_folds} ---")

        wandb.init(
            project="ResDeepSurv-BraTS",
            name=f"fold_{fold + 1}",
            group=cv_group,
            job_type="cross_val_fold",
            config={
                "fold": fold + 1,
                "num_folds": num_folds,
                "epochs": epochs,
                "batch_size": batch_size,
                "learning_rate": lr,
                "input_dim": input_dim,
            },
        )

        X_train, times_train, events_train = X_train_cv[train_idx], times_train_cv[train_idx], events_train_cv[train_idx]
        X_val,   times_val,   events_val   = X_train_cv[val_idx],   times_train_cv[val_idx],   events_train_cv[val_idx]

        y_train_sksurv = np.array(
            list(zip(events_train, times_train)),
            dtype=[("Status", "?"), ("Survival_in_days", "<f8")],
        )
        y_val_sksurv = np.array(
            list(zip(events_val, times_val)),
            dtype=[("Status", "?"), ("Survival_in_days", "<f8")],
        )

        train_dataset = SurvivalDataset(X_train, times_train, events_train.astype(np.float32))
        val_dataset   = SurvivalDataset(X_val,   times_val,   events_val.astype(np.float32))

        train_loader = DataLoader(
            train_dataset, batch_size=batch_size, shuffle=True,
            collate_fn=survival_collate, drop_last=True,
        )
        train_eval_loader = DataLoader(          # no drop_last — used for Breslow/IBS
            train_dataset, batch_size=batch_size, shuffle=False,
            collate_fn=survival_collate,
        )
        val_loader = DataLoader(
            val_dataset, batch_size=batch_size, shuffle=False,
            collate_fn=survival_collate,
        )

        model     = ResDeepSurv(input_dim=input_dim).to(device)
        optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-5)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=epochs, eta_min=1e-6
        )

        best_epoch_cindex = -1.0
        best_epoch_state  = None
        patience_counter  = 0
        patience          = 15

        for epoch in range(epochs):
            train_loss = train_epoch(model, train_loader, criterion, optimizer, device)

            val_loss,   val_risks,   val_t,   val_e   = evaluate(model, val_loader,       criterion, device)
            _,          train_risks, train_t, train_e = evaluate(model, train_eval_loader, criterion, device)

            val_c_index   = concordance_index_censored(val_e.astype(bool),   val_t,   val_risks)[0]
            train_c_index = concordance_index_censored(train_e.astype(bool), train_t, train_risks)[0]

            scheduler.step()

            time_min = max(times_train.min(), times_val.min())
            safe_time_max = min(times_train.max(), times_val.max()) - 1.0
            val_ibs = float("nan")

            if time_min < safe_time_max:
                times_for_ibs = np.linspace(time_min, safe_time_max, 100)
                b_times, b_H0 = compute_breslow_baseline_hazard(
                    train_risks, train_t, train_e
                )
                val_surv_probs = predict_survival_functions(
                    val_risks, b_times, b_H0, times_for_ibs
                )
                try:
                    val_ibs = integrated_brier_score(
                        y_train_sksurv, y_val_sksurv, val_surv_probs, times_for_ibs
                    )
                except ValueError:
                    pass

            wandb.log({
                "train_loss":    train_loss,
                "val_loss":      val_loss,
                "train_c_index": train_c_index,
                "val_c_index":   val_c_index,
                "val_ibs":       val_ibs if not np.isnan(val_ibs) else None,
                "epoch":         epoch + 1,
            })

            if (epoch + 1) % 5 == 0:
                ibs_str = f"{val_ibs:.4f}" if not np.isnan(val_ibs) else "N/A"
                print(
                    f"  Fold {fold+1} | Epoch [{epoch+1}/{epochs}] "
                    f"| Val Loss: {val_loss:.4f} "
                    f"| Val C-Index: {val_c_index:.4f} "
                    f"| Val IBS: {ibs_str}"
                )

            if val_c_index > best_epoch_cindex:
                best_epoch_cindex = val_c_index
                best_epoch_state  = copy.deepcopy(model.state_dict())
                patience_counter  = 0
            else:
                patience_counter += 1

            if patience_counter >= patience:
                print(f"  Early stopping at epoch {epoch+1} for fold {fold+1} "
                      f"(No improvement in Val C-Index for {patience} epochs)")
                break

        print(f"  Fold {fold+1} best val C-index: {best_epoch_cindex:.4f}")
        wandb.log({"best_val_c_index": best_epoch_cindex})
        wandb.finish()

        fold_model_states.append(best_epoch_state)

    print("\n--- Collecting CV risks from all fold models for ensemble Breslow ---")
    cv_dataset = SurvivalDataset(X_train_cv, times_train_cv, events_train_cv.astype(np.float32))
    cv_loader  = DataLoader(
        cv_dataset, batch_size=batch_size, shuffle=False,
        collate_fn=survival_collate,
    )

    for state in fold_model_states:
        m = ResDeepSurv(input_dim=input_dim).to(device)
        m.load_state_dict(state)
        _, cv_risks, cv_t, cv_e = evaluate(m, cv_loader, criterion, device)
        fold_cv_risks.append(cv_risks)

    ensemble_cv_risks = np.mean(fold_cv_risks, axis=0)   # shape (N_cv,)
    fold_cv_times  = cv_t
    fold_cv_events = cv_e

    y_cv_sksurv = np.array(
        list(zip(events_train_cv, times_train_cv)),
        dtype=[("Status", "?"), ("Survival_in_days", "<f8")],
    )

    # ── Ensemble inference on test set ───────────────────────────────────
    print("--- Ensemble Inference on Test Set ---")
    fold_test_risks = []

    for fold_idx, state in enumerate(fold_model_states):
        m = ResDeepSurv(input_dim=input_dim).to(device)
        m.load_state_dict(state)
        _, test_risks_i, test_t, test_e = evaluate(m, test_loader, criterion, device)
        fold_test_risks.append(test_risks_i)
        print(f"  Fold {fold_idx+1} model | Test C-Index: "
              f"{concordance_index_censored(test_e.astype(bool), test_t, test_risks_i)[0]:.4f}")

    ensemble_test_risks = np.mean(fold_test_risks, axis=0)

    print("\n--- Cross Validation Complete ---")
    test_c_index = concordance_index_censored(
        test_e.astype(bool), test_t, ensemble_test_risks
    )[0]
    print(f"Ensemble Test C-Index: {test_c_index:.4f}")

    time_min = max(times_train_cv.min(), times_test.min())
    safe_time_max = min(times_train_cv.max(), times_test.max()) - 1.0
    test_ibs = float("nan")

    if time_min < safe_time_max:
      times_for_ibs = np.linspace(time_min, safe_time_max, 100)
      b_times, b_H0 = compute_breslow_baseline_hazard(
          ensemble_cv_risks, fold_cv_times, fold_cv_events
      )
      test_surv_probs = predict_survival_functions(
          ensemble_test_risks, b_times, b_H0, times_for_ibs
      )
      try:
          test_ibs = integrated_brier_score(
              y_cv_sksurv, y_test_sksurv, test_surv_probs, times_for_ibs
          )
          print(f"Ensemble Test IBS: {test_ibs:.4f}")
      except ValueError as exc:
          print(f"Could not compute Test IBS: {exc}")

    # ── Log test results as a separate W&B run ───────────────────────────
    wandb.init(
        project="ResDeepSurv-BraTS",
        name="ensemble_test_eval",
        group=cv_group,
        job_type="test_eval",
        config={"num_folds": num_folds, "ensemble": "mean_risk"},
    )
    wandb.log({
        "test/ensemble_c_index": test_c_index,
        "test/ensemble_ibs":     test_ibs if not np.isnan(test_ibs) else None,
        **{f"test/fold_{i+1}_c_index": concordance_index_censored(
                test_e.astype(bool), test_t, fold_test_risks[i]
            )[0]
           for i in range(num_folds)},
    })
    wandb.finish()

    return {
        "test_c_index": test_c_index,
        "test_ibs":     test_ibs,
        "fold_test_risks": fold_test_risks,
        "ensemble_test_risks": ensemble_test_risks,
    }


In [ ]:
import numpy as np
from sklearn.preprocessing import StandardScaler

X_train_raw = train_df.drop(columns=['BraTS20ID', 'Survival_days']).values.astype(np.float32)
X_test_raw = test_df.drop(columns=['BraTS20ID', 'Survival_days']).values.astype(np.float32)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_raw).astype(np.float32)
X_test = scaler.transform(X_test_raw).astype(np.float32)

times_train = train_df['Survival_days'].values
events_train = pd.Series(np.ones(len(train_df)), dtype=np.float32).values

times_test = test_df['Survival_days'].values
events_test = pd.Series(np.ones(len(test_df)), dtype=np.float32).values

input_dim = X_train.shape[1]
run_cross_validation(X_train, times_train, events_train, X_test, times_test, events_test, input_dim, epochs=100, lr=5e-4)


Using device: cpu

--- Starting Fold 1/5 ---


  Fold 1 | Epoch [5/100] | Val Loss: 2.0250 | Val C-Index: 0.5712 | Val IBS: 0.1182
  Fold 1 | Epoch [10/100] | Val Loss: 2.3210 | Val C-Index: 0.6154 | Val IBS: 0.1255
  Fold 1 | Epoch [15/100] | Val Loss: 2.0330 | Val C-Index: 0.4473 | Val IBS: 0.1198
  Fold 1 | Epoch [20/100] | Val Loss: 2.0445 | Val C-Index: 0.5356 | Val IBS: 0.1321
  Fold 1 | Epoch [25/100] | Val Loss: 2.0297 | Val C-Index: 0.5641 | Val IBS: 0.1285
  Early stopping at epoch 25 for fold 1 (No improvement in Val C-Index for 15 epochs)
  Fold 1 best val C-index: 0.6154


best_val_c_index,▁
epoch,▁▁▂▂▂▂▃▃▃▄▄▄▅▅▅▅▆▆▆▇▇▇▇██
train_c_index,▁▂▃▄▄▄▅▅▆▆▆▄▃▄▄▆▆▇▆▇▇▇███
train_loss,▅▇█▅▄▄▂▃▄▇▃▅▄▄▂▃▂▃▂▃▃▁▂▃▂
val_c_index,▃▇██▆▆▆▇██▇▅▂▁▁▃▅▅▅▅▅▆▆▇▆
val_ibs,█▅▃▂▂▁▂▂▂▃▂▁▁▂▂▂▃▃▄▄▄▄▄▄▃
val_loss,▂▁▂▃▃▄▅▇██▆▃▄▅▃▃▃▃▃▃▃▃▃▃▃
best_val_c_index,0.61538
epoch,25
train_c_index,0.69441
train_loss,3.09152



--- Starting Fold 2/5 ---


  Fold 2 | Epoch [5/100] | Val Loss: 2.3221 | Val C-Index: 0.6373 | Val IBS: 0.2737
  Fold 2 | Epoch [10/100] | Val Loss: 2.3718 | Val C-Index: 0.6558 | Val IBS: 0.2615
  Fold 2 | Epoch [15/100] | Val Loss: 2.3042 | Val C-Index: 0.6629 | Val IBS: 0.2295
  Early stopping at epoch 16 for fold 2 (No improvement in Val C-Index for 15 epochs)
  Fold 2 best val C-index: 0.6700


...

  Fold 5 | Epoch [5/100] | Val Loss: 2.5462 | Val C-Index: 0.4625 | Val IBS: 0.1736
  Fold 5 | Epoch [10/100] | Val Loss: 2.0984 | Val C-Index: 0.5225 | Val IBS: 0.2075
  Fold 5 | Epoch [15/100] | Val Loss: 2.1967 | Val C-Index: 0.4805 | Val IBS: 0.1816
  Fold 5 | Epoch [20/100] | Val Loss: 2.1260 | Val C-Index: 0.4955 | Val IBS: 0.1725
  Early stopping at epoch 24 for fold 5 (No improvement in Val C-Index for 15 epochs)
  Fold 5 best val C-index: 0.5285


best_val_c_index,▁
epoch,▁▁▂▂▂▃▃▃▃▄▄▄▅▅▅▆▆▆▆▇▇▇██
train_c_index,▁▂▃▄▅▅▆▆▆▆▆▆▇▇▇▇▇███████
train_loss,▇▆█▆▇▅▇▆▆▅▄▅▄▄▄▃▃▄▁▂▂▂▃▃
val_c_index,▆▂▁▂▃▄▆███▆▆▆▆▅▄▅▅▆▆▅▅▆▅
val_ibs,▁▁▂▃▅▆▇███▇▇▇▆▆▅▅▄▅▅▅▄▄▄
val_loss,▂▄▇█▆▆▄▃▂▂▂▃▃▃▃▃▃▃▂▂▂▂▁▁
best_val_c_index,0.52853
epoch,24
train_c_index,0.7326
train_loss,3.15431



--- Collecting CV risks from all fold models for ensemble Breslow ---
--- Ensemble Inference on Test Set ---
  Fold 1 model | Test C-Index: 0.5527
  Fold 2 model | Test C-Index: 0.4607
  Fold 3 model | Test C-Index: 0.5953
  Fold 4 model | Test C-Index: 0.5555
  Fold 5 model | Test C-Index: 0.5305

--- Cross Validation Complete ---
Ensemble Test C-Index: 0.5606
Ensemble Test IBS: 0.2728


test/ensemble_c_index,▁
test/ensemble_ibs,▁
test/fold_1_c_index,▁
test/fold_2_c_index,▁
test/fold_3_c_index,▁
test/fold_4_c_index,▁
test/fold_5_c_index,▁
test/ensemble_c_index,0.56059
test/ensemble_ibs,0.27283
test/fold_1_c_index,0.55273
test/fold_2_c_index,0.46068


{'test_c_index': np.float64(0.5605920444033302),
 'test_ibs': np.float64(0.27283105421328335),
 'fold_test_risks': [array([ -1.2585404 ,  -1.6171914 ,   0.09499376,  -1.6299592 ,
          -1.2620655 ,  -1.0693848 ,  -1.3990521 ,  -1.4494886 ,
          -0.8513793 ,  -1.3705027 ,  -1.1882203 ,  -1.0687147 ,
          -0.7340749 ,  -1.0673676 ,  -1.3265189 ,  -1.3140484 ,
          -1.99332   ,  -1.5700341 ,  -1.5241685 ,  -1.0099623 ,
          -1.5326493 ,  -0.9990924 ,  -1.8437043 ,  -1.24452   ,
          -1.5700489 ,  -1.2947581 ,  -1.5124669 ,  -0.49379283,
          -1.9674343 ,  -1.3470924 ,  -0.6851049 ,  -0.87644064,
         -21.550705  , -21.550705  , -21.550705  , -21.550705  ,
         -21.550705  , -21.550705  , -21.550705  , -21.550705  ,
         -21.550705  , -21.550705  , -21.550705  , -21.550705  ,
         -21.550705  , -21.550705  , -21.550705  ], dtype=float32),
  array([2.0588624, 1.4250221, 2.843294 , 1.7873111, 2.4610982, 1.7423165,
         2.0043142, 2.164270

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sksurv.linear_model import CoxPHSurvivalAnalysis
from sksurv.metrics import concordance_index_censored, integrated_brier_score

def prepare_data_for_sksurv(df):
    events = np.ones(len(df), dtype=bool)
    times = df['Survival_days'].values.astype(np.float64)
    y = np.array(list(zip(events, times)), dtype=[('Status', '?'), ('Survival_in_days', '<f8')])

    X = df.drop(columns=['BraTS20ID', 'Survival_days'])
    X = X.astype(np.float64)

    return X, y

def run_cox_baseline():
    print("--- KHỞI CHẠY MÔ HÌNH COX PROPORTIONAL HAZARDS BASELINE ---")

    X_train, y_train = prepare_data_for_sksurv(train_df)
    X_test, y_test = prepare_data_for_sksurv(test_df)

    scaler = StandardScaler()
    X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns)
    X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns)

    estimator = CoxPHSurvivalAnalysis(alpha=1e-4)
    estimator.fit(X_train_scaled, y_train)

    print("\n--- KẾT QUẢ ĐÁNH GIÁ (C-INDEX) ---")
    print(f"C-index trên tập Train: {estimator.score(X_train_scaled, y_train):.4f}")
    print(f"C-index trên tập Test:  {estimator.score(X_test_scaled, y_test):.4f}")

    print("\n--- KẾT QUẢ ĐÁNH GIÁ (IBS) ---")

    time_min = max(y_train['Survival_in_days'].min(), y_test['Survival_in_days'].min())
    time_max_test = y_test['Survival_in_days'].max()
    time_max_train = y_train['Survival_in_days'].max()
    safe_time_max = min(time_max_test, time_max_train) - 1.0

    times_for_ibs = np.linspace(time_min, safe_time_max, 100)

    try:
        surv_funcs = estimator.predict_survival_function(X_test_scaled)
        probs = np.asarray([[fn(t) for t in times_for_ibs] for fn in surv_funcs])

        ibs_value = integrated_brier_score(y_train, y_test, probs, times_for_ibs)
        print(f"Thời gian đánh giá IBS từ: {time_min:.1f} đến {safe_time_max:.1f} ngày.")
        print(f"Integrated Brier Score (IBS) trên tập Test: {ibs_value:.4f}")

    except ValueError as e:
        print(f"\n[LỖI IBS] Thông tin debug chi tiết:")
        print(f"  Max thời gian Train: {time_max_train:.1f}")
        print(f"  Max thời gian Test:  {time_max_test:.1f}")
        print(f"  Vùng chọn để tính:   {safe_time_max:.1f}")
        print(f"  Thông báo gốc:       {e}")
        print("  Giải pháp: Hãy điều chỉnh safe_time_max bằng cách trừ đi một số lớn hơn.")

if __name__ == "__main__":
    run_cox_baseline()


--- KHỞI CHẠY MÔ HÌNH COX PROPORTIONAL HAZARDS BASELINE ---

--- KẾT QUẢ ĐÁNH GIÁ (C-INDEX) ---
C-index trên tập Train: 0.7208
C-index trên tập Test:  0.6272

--- KẾT QUẢ ĐÁNH GIÁ (IBS) ---
Thời gian đánh giá IBS từ: 12.0 đến 1591.0 ngày.
Integrated Brier Score (IBS) trên tập Test: 0.1269
